In [43]:
from transformers import EncoderDecoderModel
import torch

model = EncoderDecoderModel.from_encoder_decoder_pretrained(
    "bert-base-uncased",  # encoder
    "bert-base-uncased"   # decoder
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using: {device}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertLMHeadModel LOAD REPORT from: bert-base-uncased
Key                                                                | Status     | 
-------------------------------------------------------------------+------------+-
bert.pooler.dense.weight                                           | UNEXPECTED | 
cls.seq_relationship.weight                                        | UNEXPECTED | 
bert.pooler.dense.bias                                             | UNEXPECTED | 
cls.seq_relationship.bias                                          | UNEXPECTED | 
bert.encoder.layer.{0...11}.crossattention.self.value.weight       | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.value.bias         | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.query.weight       | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.output.dense.weight     | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.output.LayerNorm.weight | MISSING    | 
bert.encoder.layer.{0...11}.crossat

using: cuda


In [44]:
import torch
import torch.nn as nn
from transformers import BertModel, BertConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using: {device}")

# 1. Encoder
class BertEncoder(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        for param in self.bert.parameters():
            param.requires_grad = False
        self.project_down = nn.Linear(768, latent_dim)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq = out.last_hidden_state        # [B, seq_len, 768] — all tokens, not just CLS
        z   = self.project_down(seq)       # [B, seq_len, latent_dim]
        return z

# 2. Parallel Decoder
class ParallelDecoder(nn.Module):
    def __init__(self, latent_dim=256, seq_len=128, vocab_size=30522):
        super().__init__()
        self.seq_len = seq_len
        self.project_up = nn.Linear(latent_dim, 768)   # per-token projection
        config = BertConfig.from_pretrained("bert-base-uncased")
        config.is_decoder = False
        self.bert      = BertModel.from_pretrained("bert-base-uncased", config=config)
        self.to_logits = nn.Linear(768, vocab_size)

    def forward(self, z):
        # z: [B, seq_len, latent_dim]
        x      = self.project_up(z)                    # [B, seq_len, 768]
        out    = self.bert(inputs_embeds=x)
        logits = self.to_logits(out.last_hidden_state)
        return logits

# 3. Test
encoder = BertEncoder(latent_dim=256).to(device)
decoder = ParallelDecoder(latent_dim=256).to(device)

dummy_ids  = torch.randint(0, 30522, (2, 128)).to(device)
dummy_mask = torch.ones(2, 128, dtype=torch.long).to(device)

with torch.no_grad():
    z      = encoder(dummy_ids, dummy_mask)
    logits = decoder(z)

print(f"z shape:      {z.shape}")       # [2, 128, 256]
print(f"logits shape: {logits.shape}")  # [2, 128, 30522]
print("forward pass ok")


using: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


z shape:      torch.Size([2, 128, 256])
logits shape: torch.Size([2, 128, 30522])
forward pass ok


In [45]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# encode a real sentence
text = "the cat sat on the mat"
inputs = tokenizer(
    text,
    return_tensors="pt",
    max_length=128,
    padding="max_length",
    truncation=True
)

# read the actual device from the model — immune to stale `device` variable
model_device   = next(encoder.parameters()).device
input_ids      = inputs["input_ids"].to(model_device)
attention_mask = inputs["attention_mask"].to(model_device)

# forward pass
with torch.no_grad():
    z = encoder(input_ids, attention_mask)
    logits = decoder(z)
    pred_ids = logits.argmax(-1)  # [1, 128]

# decode both
original  = tokenizer.decode(input_ids[0], skip_special_tokens=True)
predicted = tokenizer.decode(pred_ids[0], skip_special_tokens=True)

print(f"original:  {original}")
print(f"predicted: {predicted}")


original:  the cat sat on the mat
predicted: ##sey compliance compliancesey complianceseysey complianceseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseyseysey


In [46]:
import torch.optim as optim
from datasets import load_dataset

# load small slice first
ds = load_dataset("wikitext", "wikitext-103-raw-v1")
small_train = ds["train"].select(range(5000))  # 5000 samples to start
small_val   = ds["validation"]

print(f"train: {len(small_train)} samples")
print(f"val:   {len(small_val)} samples")

train: 5000 samples
val:   3760 samples


In [47]:
# use full dataset
full_train = ds["train"].filter(lambda x: len(x["text"].strip()) > 10)

print(f"full train samples: {len(full_train)}")

# retokenize
train_tok_full = full_train.map(tokenize, batched=True)
train_tok_full.set_format(type="torch", columns=["input_ids", "attention_mask"])

train_loader_full = DataLoader(train_tok_full, batch_size=32, shuffle=True)

print(f"full train batches: {len(train_loader_full)}")

full train samples: 1154847
full train batches: 36089


In [48]:
import torch.nn.functional as F
from torch.optim import AdamW

decoder = decoder.to(device)

# only decoder is updated — encoder is frozen and already cached
optimizer = AdamW(list(decoder.parameters()), lr=1e-4)

EPOCHS     = 3
VOCAB_SIZE = 30522
best_val_loss = float("inf")

for epoch in range(EPOCHS):

    # ── training ──────────────────────────────────────
    decoder.train()
    train_loss = 0

    for step, (z, input_ids) in enumerate(train_loader):
        z         = z.to(device)
        input_ids = input_ids.to(device)

        logits = decoder(z)

        loss = F.cross_entropy(
            logits.view(-1, VOCAB_SIZE),
            input_ids.view(-1),
            ignore_index=0
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        if step % 50 == 0:
            print(f"epoch {epoch+1} step {step}/{len(train_loader)} | loss {loss.item():.4f}")

    avg_train = train_loss / len(train_loader)

    # ── validation ────────────────────────────────────
    decoder.eval()
    val_loss = 0

    with torch.no_grad():
        for z, input_ids in val_loader:
            z         = z.to(device)
            input_ids = input_ids.to(device)

            logits = decoder(z)

            loss = F.cross_entropy(
                logits.view(-1, VOCAB_SIZE),
                input_ids.view(-1),
                ignore_index=0
            )
            val_loss += loss.item()

    avg_val = val_loss / len(val_loader)
    print(f"epoch {epoch+1} done | train loss {avg_train:.4f} | val loss {avg_val:.4f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({"decoder": decoder.state_dict()}, "stage1_best.pt")
        print(f"saved best model at val loss {best_val_loss:.4f}")


ValueError: not enough values to unpack (expected 2, got 1)

In [50]:

# fresh decoder with correct architecture, then load trained weights
decoder = ParallelDecoder(latent_dim=256).to(device)
checkpoint = torch.load("stage1_best.pt", map_location=device, weights_only=True)
decoder.load_state_dict(checkpoint["decoder"])
decoder.eval()

text = "the cat sat on the mat"
inputs = tokenizer(
    text,
    return_tensors="pt",
    max_length=128,
    padding="max_length",
    truncation=True
)

model_device   = next(encoder.parameters()).device
input_ids      = inputs["input_ids"].to(model_device)
attention_mask = inputs["attention_mask"].to(model_device)

with torch.no_grad():
    z        = encoder(input_ids, attention_mask)
    logits   = decoder(z)
    pred_ids = logits.argmax(-1)

original  = tokenizer.decode(input_ids[0],  skip_special_tokens=True)
predicted = tokenizer.decode(pred_ids[0].cpu(), skip_special_tokens=True)

print(f"original:  {original}")
print(f"predicted: {predicted}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RuntimeError: Error(s) in loading state_dict for ParallelDecoder:
	size mismatch for project_up.weight: copying a param with shape torch.Size([98304, 256]) from checkpoint, the shape in current model is torch.Size([768, 256]).
	size mismatch for project_up.bias: copying a param with shape torch.Size([98304]) from checkpoint, the shape in current model is torch.Size([768]).